In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# =========================================================
# 0. 基本設定
# =========================================================
poi_lat = 25.0190099143756
poi_lon = 121.53137177116402

# ===== 路徑 =====
tmean_gwl_folder = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/GWL1.5")
tmax_gwl_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/GWL1.5")
tmin_gwl_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/GWL1.5")

tmean_hist_folder = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/historical")
tmax_hist_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/historical")
tmin_hist_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/historical")

save_path = Path("/Users/stevechiao/Climate_project\程式作業\figue")
save_path.mkdir(parents=True, exist_ok=True)

In [12]:
# =========================================================
# 1. 基本工具
# =========================================================
def safe_filename(text):
    return (
        text.replace(" ", "_")
            .replace(">", "gt")
            .replace("<", "lt")
            .replace("=", "eq")
            .replace("°", "")
            .replace("/", "_")
    )


def read_csv_with_fallback(csv_path: Path):
    encodings = ["utf-8", "utf-8-sig", "big5"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(csv_path, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error

In [13]:
# ===== 模式與情境 =====
model_name = "ACCESS-CM2"
target_ssps = ["ssp245", "ssp370", "ssp585"]

# ===== 時間範圍 =====
hist_start = "2004-01-01"
hist_end   = "2014-12-31"
fut_start  = "2015-01-01"
fut_end    = "2034-12-31"

start_year = 2004
end_year = 2034

INVALID_VALUE = -99.9

ssp_colors = {
    "ssp245": "#2ca02c",  # green
    "ssp370": "#ff7f0e",  # orange
    "ssp585": "#d62728",  # red
}

season_order = ["DJF", "MAM", "JJA", "SON"]


# =========================================================
# 1. 基本工具
# =========================================================
def safe_filename(text):
    return (
        text.replace(" ", "_")
            .replace(">", "gt")
            .replace("<", "lt")
            .replace("=", "eq")
            .replace("°", "")
            .replace("/", "_")
    )


def read_csv_with_fallback(csv_path: Path):
    encodings = ["utf-8", "utf-8-sig", "big5"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(csv_path, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error


def extract_year_from_filename(fp: Path):
    m = re.search(r"_(\d{4})\.csv$", fp.name)
    return int(m.group(1)) if m else -1


def get_historical_files(folder: Path, model_name: str):
    files = sorted(folder.glob("*.csv"))
    matched = [f for f in files if model_name in f.name]
    return sorted(matched, key=extract_year_from_filename)


def get_gwl_ssp_files(folder: Path, model_name: str, ssp_name: str):
    files = sorted(folder.glob("*.csv"))
    matched = [f for f in files if (model_name in f.name) and (ssp_name in f.name)]
    return sorted(matched, key=extract_year_from_filename)


def read_point_daily_series_from_csv(csv_path: Path, poi_lon: float, poi_lat: float):
    """
    格式：
    前兩欄：經緯度
    後面欄位：YYYYMMDD
    """
    df = read_csv_with_fallback(csv_path)

    cols_upper = [str(c).upper() for c in df.columns]
    df.columns = cols_upper

    lon_col = df.columns[0]
    lat_col = df.columns[1]

    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")

    dist = np.sqrt((df[lon_col] - poi_lon) ** 2 + (df[lat_col] - poi_lat) ** 2)
    idx = dist.idxmin()
    row = df.loc[idx]

    date_cols = df.columns[2:]
    dates = pd.to_datetime(date_cols, format="%Y%m%d", errors="coerce")
    values = pd.to_numeric(row[date_cols], errors="coerce").replace(INVALID_VALUE, np.nan)

    s = pd.Series(values.values, index=dates)
    s = s[~s.index.isna()].sort_index()

    return s, float(row[lon_col]), float(row[lat_col])


def load_daily_series_from_files(files, poi_lon, poi_lat):
    if len(files) == 0:
        raise FileNotFoundError("找不到檔案")

    series_list = []
    nearest_lon = None
    nearest_lat = None

    for f in files:
        s, lon0, lat0 = read_point_daily_series_from_csv(f, poi_lon, poi_lat)
        series_list.append(s)
        nearest_lon = lon0
        nearest_lat = lat0

    full = pd.concat(series_list, axis=0)
    full = full[~full.index.duplicated(keep="first")]
    full = full.sort_index()

    return full, nearest_lon, nearest_lat


def load_historical_series(folder, model_name, poi_lon, poi_lat):
    files = get_historical_files(folder, model_name)
    return load_daily_series_from_files(files, poi_lon, poi_lat)


def load_gwl_ssp_series(folder, model_name, ssp_name, poi_lon, poi_lat):
    files = get_gwl_ssp_files(folder, model_name, ssp_name)
    return load_daily_series_from_files(files, poi_lon, poi_lat)


def combine_historical_and_ssp(hist_series, ssp_series):
    hist_part = hist_series[hist_start:hist_end]
    ssp_part  = ssp_series[fut_start:fut_end]

    combined = pd.concat([hist_part, ssp_part])
    combined = combined.sort_index()
    combined = combined[~combined.index.duplicated(keep="first")]

    return combined


# =========================================================
# 2. 月統計與季統計
# =========================================================
def calc_monthly_stats(tmean_series, tmax_series, tmin_series):
    df = pd.DataFrame({
        "tmean": tmean_series,
        "tmax": tmax_series,
        "tmin": tmin_series
    }).dropna(how="all").sort_index()

    df = df[(df.index.year >= start_year) & (df.index.year <= end_year)].copy()

    df["year"] = df.index.year
    df["month"] = df.index.month
    df["ym"] = df.index.to_period("M")

    monthly = df.groupby("ym").agg(
        tmean_monthly_mean=("tmean", "mean"),
        tmax_gt31_days=("tmax", lambda x: (x > 31).sum()),
        tmax_ge35_days=("tmax", lambda x: (x >= 35).sum()),
        tmax_ge40_days=("tmax", lambda x: (x >= 40).sum()),
        tmin_lt15_9_days=("tmin", lambda x: (x < 15.9).sum()),
        n_days=("tmean", "count"),
    )

    monthly["year"] = monthly.index.year
    monthly["month"] = monthly.index.month
    return monthly.reset_index(drop=False)


def month_to_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"


def assign_season_year(year, month):
    if month == 12:
        return year + 1
    return year


def calc_seasonal_stats_from_monthly(monthly_df):
    df = monthly_df.copy()
    df["season"] = df["month"].apply(month_to_season)
    df["season_year"] = df.apply(lambda r: assign_season_year(int(r["year"]), int(r["month"])), axis=1)

    seasonal = df.groupby(["season_year", "season"]).agg(
        tmean_seasonal_mean=("tmean_monthly_mean", "mean"),
        tmax_gt31_days=("tmax_gt31_days", "sum"),
        tmax_ge35_days=("tmax_ge35_days", "sum"),
        tmax_ge40_days=("tmax_ge40_days", "sum"),
        tmin_lt15_9_days=("tmin_lt15_9_days", "sum"),
        n_days=("n_days", "sum"),
    ).reset_index()

    seasonal["season"] = pd.Categorical(seasonal["season"], categories=season_order, ordered=True)
    seasonal = seasonal.sort_values(["season_year", "season"])

    return seasonal


# =========================================================
# 3. 主流程：單一模式、不同 SSP
# =========================================================
monthly_results = {}
seasonal_results = {}
grid_info = {}

for ssp in target_ssps:
    try:
        hist_tmean, lon0, lat0 = load_historical_series(tmean_hist_folder, model_name, poi_lon, poi_lat)
        hist_tmax, _, _ = load_historical_series(tmax_hist_folder, model_name, poi_lon, poi_lat)
        hist_tmin, _, _ = load_historical_series(tmin_hist_folder, model_name, poi_lon, poi_lat)

        ssp_tmean, _, _ = load_gwl_ssp_series(tmean_gwl_folder, model_name, ssp, poi_lon, poi_lat)
        ssp_tmax, _, _ = load_gwl_ssp_series(tmax_gwl_folder, model_name, ssp, poi_lon, poi_lat)
        ssp_tmin, _, _ = load_gwl_ssp_series(tmin_gwl_folder, model_name, ssp, poi_lon, poi_lat)

        tmean_series = combine_historical_and_ssp(hist_tmean, ssp_tmean)
        tmax_series  = combine_historical_and_ssp(hist_tmax,  ssp_tmax)
        tmin_series  = combine_historical_and_ssp(hist_tmin,  ssp_tmin)

        monthly_df = calc_monthly_stats(tmean_series, tmax_series, tmin_series)
        seasonal_df = calc_seasonal_stats_from_monthly(monthly_df)

        monthly_results[ssp] = monthly_df
        seasonal_results[ssp] = seasonal_df
        grid_info[ssp] = {"lon": lon0, "lat": lat0}

        print(f"[OK] {model_name} | {ssp}")
        print(f"     nearest grid = ({lat0:.4f}, {lon0:.4f})")
        print(f"     monthly rows = {len(monthly_df)}, seasonal rows = {len(seasonal_df)}")

    except Exception as e:
        print(f"[SKIP] {model_name} | {ssp}: {e}")


# =========================================================
# 4. 季節摘要表
# =========================================================
def build_seasonal_summary_table(seasonal_results, ssp_order=None, season_order=None):
    if ssp_order is None:
        ssp_order = sorted(seasonal_results.keys())
    if season_order is None:
        season_order = ["DJF", "MAM", "JJA", "SON"]

    summary = {}

    for ssp in ssp_order:
        sdf = seasonal_results[ssp].copy()
        tmp = (
            sdf.groupby("season")
            .agg(
                tmean_seasonal_mean=("tmean_seasonal_mean", "mean"),
                tmax_gt31_days=("tmax_gt31_days", "mean"),
                tmax_ge35_days=("tmax_ge35_days", "mean"),
                tmax_ge40_days=("tmax_ge40_days", "mean"),
                tmin_lt15_9_days=("tmin_lt15_9_days", "mean"),
            )
            .reindex(season_order)
        )
        summary[ssp] = tmp

    return summary


# =========================================================
# 5. 圖 1：季節長條圖
# =========================================================
def plot_seasonal_bar_charts(seasonal_results, model_name, save_path=None):
    seasonal_summary = build_seasonal_summary_table(
        seasonal_results,
        ssp_order=target_ssps,
        season_order=season_order
    )

    metrics = [
        ("tmean_seasonal_mean", "Seasonal Mean Tmean", "°C"),
        ("tmax_gt31_days", "Seasonal Days with Tmax > 31°C", "Days"),
        ("tmax_ge35_days", "Seasonal Days with Tmax ≥ 35°C", "Days"),
        ("tmin_lt15_9_days", "Seasonal Days with Tmin < 15.9°C", "Days"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    x = np.arange(len(season_order))
    width = 0.24

    for ax, (metric, title, ylabel) in zip(axes, metrics):
        for i, ssp in enumerate(target_ssps):
            vals = seasonal_summary[ssp][metric].values
            ax.bar(
                x + (i - 1) * width,
                vals,
                width=width,
                label=ssp,
                color=ssp_colors.get(ssp),
                alpha=0.85
            )

        ax.set_xticks(x)
        ax.set_xticklabels(season_order)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    fig.suptitle(f"Daxue_village - {model_name} Seasonal Cockroach Thermal Metrics (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename('Seasonal Cockroach Thermal Metrics')}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 6. 圖 2：每月閾值折線圖
# =========================================================
def build_monthly_threshold_climatology(monthly_results, ssp_order=None):
    if ssp_order is None:
        ssp_order = sorted(monthly_results.keys())

    climatology = {}
    for ssp in ssp_order:
        mdf = monthly_results[ssp].copy()
        tmp = (
            mdf.groupby("month")
            .agg(
                tmax_gt31_days=("tmax_gt31_days", "mean"),
                tmax_ge35_days=("tmax_ge35_days", "mean"),
                tmax_ge40_days=("tmax_ge40_days", "mean"),
                tmin_lt15_9_days=("tmin_lt15_9_days", "mean"),
            )
            .reindex(range(1, 13))
        )
        climatology[ssp] = tmp

    return climatology


def plot_monthly_threshold_lines(monthly_results, model_name, save_path=None):
    monthly_clim = build_monthly_threshold_climatology(monthly_results, ssp_order=target_ssps)

    metrics = [
        ("tmax_gt31_days", "Monthly Mean Days with Tmax > 31°C", "Days / Month"),
        ("tmax_ge35_days", "Monthly Mean Days with Tmax ≥ 35°C", "Days / Month"),
        ("tmax_ge40_days", "Monthly Mean Days with Tmax ≥ 40°C", "Days / Month"),
        ("tmin_lt15_9_days", "Monthly Mean Days with Tmin < 15.9°C", "Days / Month"),
    ]

    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

    for ax, (metric, title, ylabel) in zip(axes, metrics):
        for ssp in target_ssps:
            vals = monthly_clim[ssp][metric].values
            ax.plot(
                range(1, 13),
                vals,
                marker="o",
                linewidth=2,
                label=ssp,
                color=ssp_colors.get(ssp)
            )

        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

    axes[-1].set_xticks(range(1, 13))
    axes[-1].set_xticklabels(month_labels)
    axes[0].legend(ncol=3, loc="upper center")

    fig.suptitle(f"Daxue_village - {model_name} Monthly Cockroach Thermal Threshold Days (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename('Monthly Cockroach Thermal Threshold Days')}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 7. 圖 3：各季節逐年變化圖（historical 與 future 分開畫）
# =========================================================
def plot_seasonal_interannual_timeseries(
    seasonal_results,
    metric,
    title,
    ylabel,
    model_name,
    save_path=None,
    ylim=None
):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.flatten()

    for i, season in enumerate(season_order):
        ax = axes[i]

        historical_drawn = False

        for ssp in target_ssps:
            if ssp not in seasonal_results:
                continue

            sdf = seasonal_results[ssp].copy()
            sub = sdf[sdf["season"] == season].copy()
            sub = sub[(sub["season_year"] >= start_year) & (sub["season_year"] <= end_year)]

            if len(sub) == 0:
                continue

            hist_part = sub[sub["season_year"] <= 2014]
            fut_part  = sub[sub["season_year"] >= 2015]

            # historical：只畫一次灰色
            if len(hist_part) > 0 and not historical_drawn:
                ax.plot(
                    hist_part["season_year"],
                    hist_part[metric],
                    color="gray",
                    linewidth=1.8,
                    alpha=0.8,
                    label="historical"
                )
                historical_drawn = True

            # future：各 SSP 彩色
            if len(fut_part) > 0:
                ax.plot(
                    fut_part["season_year"],
                    fut_part[metric],
                    marker="o",
                    linewidth=2,
                    color=ssp_colors.get(ssp),
                    label=ssp
                )

            # 分界線
            ax.axvline(x=2015, color="red", linestyle="--", linewidth=1.5, alpha=0.8)

        ax.set_title(season)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

        if ylim is not None:
            ax.set_ylim(ylim)

        xticks = np.arange(start_year, end_year + 1, 2)
        ax.set_xticks(xticks)

    handles, labels = axes[0].get_legend_handles_labels()

    # 去重 legend
    uniq = dict(zip(labels, handles))
    fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=4)

    fig.suptitle(f"Daxue_village - {model_name} {title} (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename(title)}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 8. 執行畫圖
# =========================================================
plot_seasonal_bar_charts(
    seasonal_results=seasonal_results,
    model_name=model_name,
    save_path=save_path
)

plot_monthly_threshold_lines(
    monthly_results=monthly_results,
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmean_seasonal_mean",
    title="Seasonal Mean Tmean Interannual Variation",
    ylabel="°C",
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmax_gt31_days",
    title="Seasonal Days with Tmax > 31C Interannual Variation",
    ylabel="Days",
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmin_lt15_9_days",
    title="Seasonal Days with Tmin < 15.9C Interannual Variation",
    ylabel="Days",
    model_name=model_name,
    save_path=save_path
)

[SKIP] ACCESS-CM2 | ssp245: 找不到檔案
[SKIP] ACCESS-CM2 | ssp370: 找不到檔案
[SKIP] ACCESS-CM2 | ssp585: 找不到檔案


KeyError: 'ssp245'

In [3]:
from pathlib import Path

poi_lat = 25.0190099143756
poi_lon = 121.53137177116402

tmean_folder = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/GWL1.5")
tmax_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/GWL1.5")
tmin_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/GWL1.5")

INVALID_VALUE = -99.9
model_name = "ACCESS-CM2"
target_ssps = ["ssp126", "ssp245", "ssp370", "ssp585"]

In [4]:
tmean_hist_folder = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/historical")
tmax_hist_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/historical")
tmin_hist_folder  = Path("/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/historical")

In [14]:
print("tmean exists:", tmean_folder.exists())
print("tmax exists:", tmax_folder.exists())
print("tmin exists:", tmin_folder.exists())

print("tmean sample:", list(tmean_folder.glob("*.csv"))[:3])
print("tmax sample:", list(tmax_folder.glob("*.csv"))[:3])
print("tmin sample:", list(tmin_folder.glob("*.csv"))[:3])

tmean exists: True
tmax exists: True
tmin exists: True
tmean sample: [PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_平均溫_ssp585_NESM3_2013.csv'), PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_平均溫_ssp585_MRI-ESM2-0_2035.csv'), PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_平均溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_平均溫_ssp585_EC-Earth3-Veg-LR_2022.csv')]
tmax sample: [PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最高溫_ssp126_IITM-ESM_2043.csv'), PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最高溫_ssp370_EC-Earth3-Veg_2004.csv'), PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最高溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最高溫_ssp370_EC-Earth3-Veg_2010.csv')]
tmin sample: [PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最低溫_ssp126_EC-Earth3-Veg-LR_2030.csv'), PosixPath('/Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最低溫_ssp245_EC-Earth3-Veg-LR_2027.csv'), Posix

In [8]:

INVALID_VALUE = -99.9

ssp_colors = {
    "ssp245": "#2ca02c",  # green
    "ssp370": "#ff7f0e",  # orange
    "ssp585": "#d62728",  # red
}

season_order = ["DJF", "MAM", "JJA", "SON"]


# =========================================================
# 1. 基本工具
# =========================================================
def safe_filename(text):
    return (
        text.replace(" ", "_")
            .replace(">", "gt")
            .replace("<", "lt")
            .replace("=", "eq")
            .replace("°", "")
            .replace("/", "_")
    )


def read_csv_with_fallback(csv_path: Path):
    encodings = ["utf-8", "utf-8-sig", "big5"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(csv_path, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error


def extract_year_from_filename(fp: Path):
    m = re.search(r"_(\d{4})\.csv$", fp.name)
    return int(m.group(1)) if m else -1


def get_historical_files(folder: Path, model_name: str):
    files = sorted(folder.glob("*.csv"))
    matched = [f for f in files if model_name in f.name]
    return sorted(matched, key=extract_year_from_filename)


def get_gwl_ssp_files(folder: Path, model_name: str, ssp_name: str):
    files = sorted(folder.glob("*.csv"))
    matched = [f for f in files if (model_name in f.name) and (ssp_name in f.name)]
    return sorted(matched, key=extract_year_from_filename)


def read_point_daily_series_from_csv(csv_path: Path, poi_lon: float, poi_lat: float):
    """
    格式：
    前兩欄：經緯度
    後面欄位：YYYYMMDD
    """
    df = read_csv_with_fallback(csv_path)

    cols_upper = [str(c).upper() for c in df.columns]
    df.columns = cols_upper

    lon_col = df.columns[0]
    lat_col = df.columns[1]

    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")

    dist = np.sqrt((df[lon_col] - poi_lon) ** 2 + (df[lat_col] - poi_lat) ** 2)
    idx = dist.idxmin()
    row = df.loc[idx]

    date_cols = df.columns[2:]
    dates = pd.to_datetime(date_cols, format="%Y%m%d", errors="coerce")
    values = pd.to_numeric(row[date_cols], errors="coerce").replace(INVALID_VALUE, np.nan)

    s = pd.Series(values.values, index=dates)
    s = s[~s.index.isna()].sort_index()

    return s, float(row[lon_col]), float(row[lat_col])


def load_daily_series_from_files(files, poi_lon, poi_lat):
    if len(files) == 0:
        raise FileNotFoundError("找不到檔案")

    series_list = []
    nearest_lon = None
    nearest_lat = None

    for f in files:
        s, lon0, lat0 = read_point_daily_series_from_csv(f, poi_lon, poi_lat)
        series_list.append(s)
        nearest_lon = lon0
        nearest_lat = lat0

    full = pd.concat(series_list, axis=0)
    full = full[~full.index.duplicated(keep="first")]
    full = full.sort_index()

    return full, nearest_lon, nearest_lat


def load_historical_series(folder, model_name, poi_lon, poi_lat):
    files = get_historical_files(folder, model_name)
    return load_daily_series_from_files(files, poi_lon, poi_lat)


def load_gwl_ssp_series(folder, model_name, ssp_name, poi_lon, poi_lat):
    files = get_gwl_ssp_files(folder, model_name, ssp_name)
    return load_daily_series_from_files(files, poi_lon, poi_lat)


def combine_historical_and_ssp(hist_series, ssp_series):
    hist_part = hist_series[hist_start:hist_end]
    ssp_part  = ssp_series[fut_start:fut_end]

    combined = pd.concat([hist_part, ssp_part])
    combined = combined.sort_index()
    combined = combined[~combined.index.duplicated(keep="first")]

    return combined


# =========================================================
# 2. 月統計與季統計
# =========================================================
def calc_monthly_stats(tmean_series, tmax_series, tmin_series):
    df = pd.DataFrame({
        "tmean": tmean_series,
        "tmax": tmax_series,
        "tmin": tmin_series
    }).dropna(how="all").sort_index()

    df = df[(df.index.year >= start_year) & (df.index.year <= end_year)].copy()

    df["year"] = df.index.year
    df["month"] = df.index.month
    df["ym"] = df.index.to_period("M")

    monthly = df.groupby("ym").agg(
        tmean_monthly_mean=("tmean", "mean"),
        tmax_gt31_days=("tmax", lambda x: (x > 31).sum()),
        tmax_ge35_days=("tmax", lambda x: (x >= 35).sum()),
        tmax_ge40_days=("tmax", lambda x: (x >= 40).sum()),
        tmin_lt15_9_days=("tmin", lambda x: (x < 15.9).sum()),
        n_days=("tmean", "count"),
    )

    monthly["year"] = monthly.index.year
    monthly["month"] = monthly.index.month
    return monthly.reset_index(drop=False)


def month_to_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"


def assign_season_year(year, month):
    if month == 12:
        return year + 1
    return year


def calc_seasonal_stats_from_monthly(monthly_df):
    df = monthly_df.copy()
    df["season"] = df["month"].apply(month_to_season)
    df["season_year"] = df.apply(lambda r: assign_season_year(int(r["year"]), int(r["month"])), axis=1)

    seasonal = df.groupby(["season_year", "season"]).agg(
        tmean_seasonal_mean=("tmean_monthly_mean", "mean"),
        tmax_gt31_days=("tmax_gt31_days", "sum"),
        tmax_ge35_days=("tmax_ge35_days", "sum"),
        tmax_ge40_days=("tmax_ge40_days", "sum"),
        tmin_lt15_9_days=("tmin_lt15_9_days", "sum"),
        n_days=("n_days", "sum"),
    ).reset_index()

    seasonal["season"] = pd.Categorical(seasonal["season"], categories=season_order, ordered=True)
    seasonal = seasonal.sort_values(["season_year", "season"])

    return seasonal


# =========================================================
# 3. 主流程：單一模式、不同 SSP
# =========================================================
monthly_results = {}
seasonal_results = {}
grid_info = {}

for ssp in target_ssps:
    try:
        hist_tmean, lon0, lat0 = load_historical_series(tmean_hist_folder, model_name, poi_lon, poi_lat)
        hist_tmax, _, _ = load_historical_series(tmax_hist_folder, model_name, poi_lon, poi_lat)
        hist_tmin, _, _ = load_historical_series(tmin_hist_folder, model_name, poi_lon, poi_lat)

        ssp_tmean, _, _ = load_gwl_ssp_series(tmean_gwl_folder, model_name, ssp, poi_lon, poi_lat)
        ssp_tmax, _, _ = load_gwl_ssp_series(tmax_gwl_folder, model_name, ssp, poi_lon, poi_lat)
        ssp_tmin, _, _ = load_gwl_ssp_series(tmin_gwl_folder, model_name, ssp, poi_lon, poi_lat)

        tmean_series = combine_historical_and_ssp(hist_tmean, ssp_tmean)
        tmax_series  = combine_historical_and_ssp(hist_tmax,  ssp_tmax)
        tmin_series  = combine_historical_and_ssp(hist_tmin,  ssp_tmin)

        monthly_df = calc_monthly_stats(tmean_series, tmax_series, tmin_series)
        seasonal_df = calc_seasonal_stats_from_monthly(monthly_df)

        monthly_results[ssp] = monthly_df
        seasonal_results[ssp] = seasonal_df
        grid_info[ssp] = {"lon": lon0, "lat": lat0}

        print(f"[OK] {model_name} | {ssp}")
        print(f"     nearest grid = ({lat0:.4f}, {lon0:.4f})")
        print(f"     monthly rows = {len(monthly_df)}, seasonal rows = {len(seasonal_df)}")

    except Exception as e:
        print(f"[SKIP] {model_name} | {ssp}: {e}")


# =========================================================
# 4. 季節摘要表
# =========================================================
def build_seasonal_summary_table(seasonal_results, ssp_order=None, season_order=None):
    if ssp_order is None:
        ssp_order = sorted(seasonal_results.keys())
    if season_order is None:
        season_order = ["DJF", "MAM", "JJA", "SON"]

    summary = {}

    for ssp in ssp_order:
        sdf = seasonal_results[ssp].copy()
        tmp = (
            sdf.groupby("season")
            .agg(
                tmean_seasonal_mean=("tmean_seasonal_mean", "mean"),
                tmax_gt31_days=("tmax_gt31_days", "mean"),
                tmax_ge35_days=("tmax_ge35_days", "mean"),
                tmax_ge40_days=("tmax_ge40_days", "mean"),
                tmin_lt15_9_days=("tmin_lt15_9_days", "mean"),
            )
            .reindex(season_order)
        )
        summary[ssp] = tmp

    return summary


# =========================================================
# 5. 圖 1：季節長條圖
# =========================================================
def plot_seasonal_bar_charts(seasonal_results, model_name, save_path=None):
    seasonal_summary = build_seasonal_summary_table(
        seasonal_results,
        ssp_order=target_ssps,
        season_order=season_order
    )

    metrics = [
        ("tmean_seasonal_mean", "Seasonal Mean Tmean", "°C"),
        ("tmax_gt31_days", "Seasonal Days with Tmax > 31°C", "Days"),
        ("tmax_ge35_days", "Seasonal Days with Tmax ≥ 35°C", "Days"),
        ("tmin_lt15_9_days", "Seasonal Days with Tmin < 15.9°C", "Days"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    x = np.arange(len(season_order))
    width = 0.24

    for ax, (metric, title, ylabel) in zip(axes, metrics):
        for i, ssp in enumerate(target_ssps):
            vals = seasonal_summary[ssp][metric].values
            ax.bar(
                x + (i - 1) * width,
                vals,
                width=width,
                label=ssp,
                color=ssp_colors.get(ssp),
                alpha=0.85
            )

        ax.set_xticks(x)
        ax.set_xticklabels(season_order)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    fig.suptitle(f"Daxue_village - {model_name} Seasonal Cockroach Thermal Metrics (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename('Seasonal Cockroach Thermal Metrics')}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 6. 圖 2：每月閾值折線圖
# =========================================================
def build_monthly_threshold_climatology(monthly_results, ssp_order=None):
    if ssp_order is None:
        ssp_order = sorted(monthly_results.keys())

    climatology = {}
    for ssp in ssp_order:
        mdf = monthly_results[ssp].copy()
        tmp = (
            mdf.groupby("month")
            .agg(
                tmax_gt31_days=("tmax_gt31_days", "mean"),
                tmax_ge35_days=("tmax_ge35_days", "mean"),
                tmax_ge40_days=("tmax_ge40_days", "mean"),
                tmin_lt15_9_days=("tmin_lt15_9_days", "mean"),
            )
            .reindex(range(1, 13))
        )
        climatology[ssp] = tmp

    return climatology


def plot_monthly_threshold_lines(monthly_results, model_name, save_path=None):
    monthly_clim = build_monthly_threshold_climatology(monthly_results, ssp_order=target_ssps)

    metrics = [
        ("tmax_gt31_days", "Monthly Mean Days with Tmax > 31°C", "Days / Month"),
        ("tmax_ge35_days", "Monthly Mean Days with Tmax ≥ 35°C", "Days / Month"),
        ("tmax_ge40_days", "Monthly Mean Days with Tmax ≥ 40°C", "Days / Month"),
        ("tmin_lt15_9_days", "Monthly Mean Days with Tmin < 15.9°C", "Days / Month"),
    ]

    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

    for ax, (metric, title, ylabel) in zip(axes, metrics):
        for ssp in target_ssps:
            vals = monthly_clim[ssp][metric].values
            ax.plot(
                range(1, 13),
                vals,
                marker="o",
                linewidth=2,
                label=ssp,
                color=ssp_colors.get(ssp)
            )

        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

    axes[-1].set_xticks(range(1, 13))
    axes[-1].set_xticklabels(month_labels)
    axes[0].legend(ncol=3, loc="upper center")

    fig.suptitle(f"Daxue_village - {model_name} Monthly Cockroach Thermal Threshold Days (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename('Monthly Cockroach Thermal Threshold Days')}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 7. 圖 3：各季節逐年變化圖（historical 與 future 分開畫）
# =========================================================
def plot_seasonal_interannual_timeseries(
    seasonal_results,
    metric,
    title,
    ylabel,
    model_name,
    save_path=None,
    ylim=None
):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.flatten()

    for i, season in enumerate(season_order):
        ax = axes[i]

        historical_drawn = False

        for ssp in target_ssps:
            if ssp not in seasonal_results:
                continue

            sdf = seasonal_results[ssp].copy()
            sub = sdf[sdf["season"] == season].copy()
            sub = sub[(sub["season_year"] >= start_year) & (sub["season_year"] <= end_year)]

            if len(sub) == 0:
                continue

            hist_part = sub[sub["season_year"] <= 2014]
            fut_part  = sub[sub["season_year"] >= 2015]

            # historical：只畫一次灰色
            if len(hist_part) > 0 and not historical_drawn:
                ax.plot(
                    hist_part["season_year"],
                    hist_part[metric],
                    color="gray",
                    linewidth=1.8,
                    alpha=0.8,
                    label="historical"
                )
                historical_drawn = True

            # future：各 SSP 彩色
            if len(fut_part) > 0:
                ax.plot(
                    fut_part["season_year"],
                    fut_part[metric],
                    marker="o",
                    linewidth=2,
                    color=ssp_colors.get(ssp),
                    label=ssp
                )

            # 分界線
            ax.axvline(x=2015, color="red", linestyle="--", linewidth=1.5, alpha=0.8)

        ax.set_title(season)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

        if ylim is not None:
            ax.set_ylim(ylim)

        xticks = np.arange(start_year, end_year + 1, 2)
        ax.set_xticks(xticks)

    handles, labels = axes[0].get_legend_handles_labels()

    # 去重 legend
    uniq = dict(zip(labels, handles))
    fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=4)

    fig.suptitle(f"Daxue_village - {model_name} {title} (2010–2034)", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_path is not None:
        filename = f"Daxue_village_{model_name}_{safe_filename(title)}_2010_2034.png"
        save_file = save_path / filename
        plt.savefig(save_file, dpi=300, bbox_inches="tight")
        print(f"✅ Saved: {save_file}")

    plt.show()


# =========================================================
# 8. 執行畫圖
# =========================================================
plot_seasonal_bar_charts(
    seasonal_results=seasonal_results,
    model_name=model_name,
    save_path=save_path
)

plot_monthly_threshold_lines(
    monthly_results=monthly_results,
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmean_seasonal_mean",
    title="Seasonal Mean Tmean Interannual Variation",
    ylabel="°C",
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmax_gt31_days",
    title="Seasonal Days with Tmax > 31C Interannual Variation",
    ylabel="Days",
    model_name=model_name,
    save_path=save_path
)

plot_seasonal_interannual_timeseries(
    seasonal_results=seasonal_results,
    metric="tmin_lt15_9_days",
    title="Seasonal Days with Tmin < 15.9C Interannual Variation",
    ylabel="Days",
    model_name=model_name,
    save_path=save_path
)

[SKIP] ACCESS-CM2 | ssp126: 找不到檔案
[SKIP] ACCESS-CM2 | ssp245: 找不到檔案
[SKIP] ACCESS-CM2 | ssp370: 找不到檔案
[SKIP] ACCESS-CM2 | ssp585: 找不到檔案


KeyError: 'ssp126'

In [9]:
csv_files = list(tmin_folder.rglob("*.csv"))
print("csv 數量：", len(csv_files))

if csv_files:
    test_file = csv_files[0]
    print("測試讀取：", test_file)

    df = read_csv_with_fallback(test_file)
    print(df.head())
    print(df.shape)
else:
    print("找不到 csv 檔")

csv 數量： 1940
測試讀取： /Users/stevechiao/AR6_統計降尺度_日資料_臺北市_最低溫/GWL1.5/AR6_統計降尺度_日資料_臺北市_最低溫_ssp126_EC-Earth3-Veg-LR_2030.csv
      LON    LAT  20300101  20300102  20300103  20300104  20300105  20300106  \
0  121.46  25.10   15.9949   15.4258   14.7385   16.1297   17.2526   18.7812   
1  121.46  25.11   14.8329   15.3760   14.7028   16.7402   17.9076   18.4996   
2  121.46  25.12   16.4886   15.4887   14.7798   16.0654   17.8157   18.6963   
3  121.46  25.13   16.4300   15.4175   14.6786   15.8223   17.7445   18.6307   
4  121.47  25.10   16.6190   14.7085   14.5145   16.0753   17.0739   19.0450   

   20300107  20300108  ...  20301223  20301224  20301225  20301226  20301227  \
0   13.6261   17.0702  ...   17.1263   15.2920   16.2622   13.5676   14.7292   
1   13.6801   16.4774  ...   17.2984   15.8293   15.3497   13.4471   15.5951   
2   13.6699   16.6398  ...   17.5893   16.7033   16.1100   12.3028   14.9251   
3   13.2018   16.7944  ...   16.7709   15.5536   16.1221   12.2491   14.6271  